# Day 2-02｜YOLO26 / Ultralytics Detection Preview

> Python「黑客松」實戰分析籃球運動數據  
> Notebook 以初學 Python 學員為主：先觀察、再改變少量變數、最後才串接。

目標：如果老師提供模型權重，就跑模型；如果沒有權重，就用 sample detection JSON 先體驗輸出格式。

In [ ]:
from pathlib import Path
import sys

# 在 Colab 中先掛載 Drive；本機執行時會自動略過。
try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception:
    pass

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機或 zip 解壓後執行時，從目前資料夾往上找。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

print("COURSE_ROOT =", COURSE_ROOT)
print("assets =", COURSE_ROOT / "assets")

In [ ]:
# 第一次需要時再打開安裝；課堂如果 Colab 環境已裝好，可以不用跑。
# !pip install -q ultralytics supervision

In [ ]:
from pathlib import Path
from src.cv_utils import read_image_rgb, draw_boxes, show_image, load_json, save_json

IMAGE_PATH = COURSE_ROOT / "assets" / "samples" / "sample_court_frame.png"
MODEL_PATH = COURSE_ROOT / "models" / "detectors" / "yolo26n_basketball_player_best.pt"
image = read_image_rgb(IMAGE_PATH)

detections = []

if MODEL_PATH.exists():
    from ultralytics import YOLO

    model = YOLO(str(MODEL_PATH))
    results = model.predict(str(IMAGE_PATH), conf=0.25, verbose=False)[0]
    names = results.names
    for box in results.boxes:
        xyxy = box.xyxy[0].cpu().numpy().tolist()
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        detections.append(
            {"bbox_xyxy": xyxy, "class_name": names[cls_id], "confidence": conf}
        )
    print("使用老師提供的 YOLO 權重。")
else:
    data = load_json(
        COURSE_ROOT / "assets" / "samples" / "sample_detections_frame0.json"
    )
    detections = data["detections"]
    print("找不到模型權重，使用 sample detections JSON。")

print("detections:", len(detections))

In [ ]:
# 只畫 confidence 較高的前幾個，避免畫面太亂。
keep = [d for d in detections if d.get("confidence", 0) >= 0.5][:12]
boxes = [d["bbox_xyxy"] for d in keep]
labels = [f"{d['class_name']} {d['confidence']:.2f}" for d in keep]
vis = draw_boxes(image, boxes, labels)
show_image(vis, "detection preview")

out_json = COURSE_ROOT / "assets" / "results" / "d2_02_detection_preview.json"
save_json({"detections": keep}, out_json)
print("saved:", out_json)

觀察問題：哪些人被漏掉？球是否太小？number box 是否容易被誤判？